# Blif Generator

This code generate blif files using yosys and ABC.

In [1]:
import os
from jinja2 import Template

In [3]:
N    = 8
PATH = f'{os.getcwd()}/results/blif_generator'
# Verify if the directory exists
if not os.path.exists(PATH):
    os.makedirs(PATH)





## Generate .pla file

In [3]:
def generate_pla(n):
    """
    Gera a tabela verdade no formato PLA para um somador de dois números de n bits.

    :param n: Número de bits de cada número de entrada.
    """
    num_inputs = 2 * n  # Número total de bits de entrada (dois números de n bits)
    num_outputs = n + 1  # Número total de bits de saída (n bits + carry out)

    # Cabeçalho do formato PLA
    pla = [f".i {num_inputs}", f".o {num_outputs}"]

    # Itera sobre todas as combinações de entrada
    for i in range(2 ** num_inputs):
        # Converte o índice para uma sequência binária de entrada
        inputs = f"{i:0{num_inputs}b}"

        # Divide os bits em dois números
        a = int(inputs[:n], 2)
        b = int(inputs[n:], 2)

        # Calcula a soma dos dois números
        total = a + b

        # Converte o resultado para uma sequência binária de saída
        outputs = f"{total:0{num_outputs}b}"

        # Adiciona a linha de entrada e saída ao PLA
        pla.append(f"{inputs} {outputs}")

    # Retorna o PLA como uma string
    pla.append(".e")
    return "\n".join(pla)

# Número de bits dos números de entrada
pla_output = generate_pla(N)

# Salva o resultado em um arquivo
with open(f"{PATH}/somador.pla", "w") as file:
    file.write(pla_output)


## Generate .blif file using yosys



In [4]:

# Template do arquivo ABC
with open(f"/app/templates/read_pla.jinja") as file:
    template_yosys = Template(file.read())

with open(f"/app/templates/abc_sript.jinja") as file:
    template_abc = Template(file.read())

# Renderiza o template com o PLA gerado
abc_script = template_abc.render(pla_file=f"{PATH}/somador.pla",
                                otimizations= ["resyn3", 'resyn'],
                                lut_k = 5,
                                verilog_output=f"{PATH}/somador.v",
                                dot_output=f"{PATH}/somador.dot",
                                blif_output=f"{PATH}/somador.blif")
                                
with open(f"{PATH}/abc_script.rc", "w") as file:
    file.write(abc_script)

yosys_script = template_yosys.render(pla=f"{PATH}/somador.pla",
                                    abc_script=f"{PATH}/abc_script.abc",
                                    verilog_output=f"{PATH}/somador.v",
                                    dot_output=f"{PATH}/somador.dot",
                                    blif_output=f"{PATH}/somador.blif")

with open(f"{PATH}/yosys_script.ys", "w") as file:
    file.write(yosys_script)

# Executa o script do Yosys

os.system(f"yosys-abc -f {PATH}/abc_script.rc")






ABC command line: "source /app/results/blif_generator/abc_script.rc".

/app/results/blif_generator/somador: i/o =   16/    9  lat =    0  and =   5073  lev = 31
/app/results/blif_generator/somador: i/o =   16/    9  lat =    0  and =   4649  lev = 30
/app/results/blif_generator/somador: i/o =   16/    9  lat =    0  nd =  1408  edge =   6017  aig  =  7215  lev = 9
/app/results/blif_generator/somador: i/o =   16/    9  lat =    0  nd =  1408  edge =   6017  bdd  =  8436  lev = 9
Limpeza do circuito concluída 
The set has more than 500 nodes. DOT file is not written.


0

## Evalutation

In [5]:
def gerar_testbench(n):
    """
    Gera um testbench Verilog parametrizado para testar o módulo `somador`.

    Parâmetros:
        n (int): Número de bits de cada operando (A e B).

    Retorna:
        str: Código Verilog do testbench.
    """
    # Verificar se o valor de n é válido
    if n < 1:
        raise ValueError("O número de bits deve ser pelo menos 1.")

    # Identificar as variáveis de entrada e saída
    inputs = [f"x{i}" for i in range(2 * n)]
    outputs = [f"z{i}" for i in range(n + 1)]

    # Cabeçalho do testbench
    testbench = f"""`include "{PATH}/somador.v"

module tb_somador;

  // Entradas
  reg {', '.join(inputs)};

  // Saídas
  wire {', '.join(outputs)};

  // Instância do módulo a ser testado
  \/app/results/blif_generator/somador uut (
    .{', .'.join([f'{inp}({inp})' for inp in inputs + outputs])}
  );

  integer A, B, C;

  initial begin
    $display(\"Iniciando os testes...\");

    // Testando todas as combinações de A e B
    for (A = 0; A < {1 << n}; A = A + 1) begin
      for (B = 0; B < {1 << n}; B = B + 1) begin
        // Configurar os bits de entrada
"""

    # Adicionar as atribuições para cada bit de A e B
    for i in range(n):
        testbench += f"        x{i} = A[{n - 1 - i}];\n"
    for i in range(n):
        testbench += f"        x{n + i} = B[{n - 1 - i}];\n"

    # Continuar o restante do código
    testbench += f"""
        // Aguardar propagação
        #10;

        // Calcular a soma esperada
        C = A + B;

        // Verificar o resultado
        if (C !== {{ {', '.join((outputs))} }}) begin
          $display(\"Erro: A=%b, B=%b -> Esperado=%b, Obtido=%b\", A, B, C, {{ {', '.join(reversed(outputs))} }});
        end
      end
    end

    $display(\"Testes concluídos.\");
    $finish;
  end
endmodule
"""

    return testbench

# Exemplo de uso
codigo_testbench = gerar_testbench(N)

# Salvar o testbench em um arquivo
with open(f"{PATH}/tb_somador.v", "w") as arquivo:
    arquivo.write(codigo_testbench)

print("Testbench gerado com sucesso!")


Testbench gerado com sucesso!


<>:44: SyntaxWarning: invalid escape sequence '\/'
<>:44: SyntaxWarning: invalid escape sequence '\/'
/tmp/ipykernel_80868/194710385.py:44: SyntaxWarning: invalid escape sequence '\/'
  """


In [6]:
# Run the testbench
os.system(f"iverilog -o {PATH}/tb_somador.vvp {PATH}/tb_somador.v")
# Get time of execution
os.system(f"vvp {PATH}/tb_somador.vvp")

Error: VVP input file 11.0 can not be run with run time version 12.0 (stable)


/app/results/blif_generator/tb_somador.v:12: error: port ``x0'' is not a port of uut.
/app/results/blif_generator/tb_somador.v:12: error: port ``x1'' is not a port of uut.
/app/results/blif_generator/tb_somador.v:12: error: port ``x2'' is not a port of uut.
/app/results/blif_generator/tb_somador.v:12: error: port ``x3'' is not a port of uut.
/app/results/blif_generator/tb_somador.v:12: error: port ``x4'' is not a port of uut.
/app/results/blif_generator/tb_somador.v:12: error: port ``x5'' is not a port of uut.
/app/results/blif_generator/tb_somador.v:12: error: port ``x6'' is not a port of uut.
/app/results/blif_generator/tb_somador.v:12: error: port ``x7'' is not a port of uut.
/app/results/blif_generator/tb_somador.v:12: error: port ``x8'' is not a port of uut.
/app/results/blif_generator/tb_somador.v:12: error: port ``x9'' is not a port of uut.
10 error(s) during elaboration.


256

In [3]:
import os

from src.ultils.sumConstructor import RippleAdderGenerator


N    = 10
PATH = f'{os.getcwd()}/results/blif_generator'
# Verify if the directory exists
if not os.path.exists(PATH):
    os.makedirs(PATH)

generator = RippleAdderGenerator(N, debug=True)
generator.save_files(path=PATH)



[DEBUG] Initializing 10-bit Ripple Adder Generator
[DEBUG] Starting file generation process
[DEBUG] Generating ripple_adder_10bit.v
[DEBUG] Generating adder Verilog code
[SUCCESS] Generated ripple_adder_10bit.v
[DEBUG] Generating ripple_adder_tb.v
[DEBUG] Generating testbench Verilog code
[SUCCESS] Generated ripple_adder_tb.v
[DEBUG] Generating optimized testbench
[DEBUG] Generating optimized testbench
[SUCCESS] Generated optimized testbench
[DEBUG] Generating exhaustive testbench
[DEBUG] Generating exhaustive testbench Verilog code
[SUCCESS] Generated exhaustive testbench
[INFO] All files generated successfully for 10-bit Ripple Adder


In [20]:
import os
import re
from itertools import product
from src.ultils.style import Mensage

class VerilogTestbenchGenerator:
    """
    Class to generate Verilog testbench for a given module.

    Attributes:
        module_path (str): Path to the source Verilog module
        module_name (str): Name of the module
        inputs (List[str]): List of input port names
        outputs (List[str]): List of output port names
        debug (bool): Flag to enable debug messages
        style (Mensage): Style class for colored output
    """

    def __init__(self, module_path: str, debug: bool = False):
        """
        Initialize the testbench generator.

        Args:
            module_path (str): Path to the source Verilog module
            debug (bool): Enable debug messages (default: False)
        """
        self.module_path = module_path
        self.module_name = ""
        self.dut_name = ""  # Actual module name in the design
        self.inputs = []
        self.outputs = []
        self.debug = debug
        self.mensage = Mensage()

        # Initial setup
        self._parse_verilog_module()

        # Initial debug message
        if self.debug:
            self.mensage.debug(f"Initializing Testbench Generator for {self.module_name}")

    def _parse_verilog_module(self):
        """
        Parse the Verilog module to extract module name, inputs, and outputs.
        """
        try:
            with open(self.module_path, 'r') as f:
                content = f.read()

            # Extract module name
            module_match = re.search(r'module\s+(\w+)\s*\(', content)
            if module_match:
                self.dut_name = module_match.group(1)
                self.module_name = f"{self.dut_name}_tb"

            # Process inputs
            input_matches = re.findall(r'input\s+([^;]+);', content)
            self.inputs = []
            for match in input_matches:
                # Split by spaces and filter out empty strings
                inputs = [inp.strip() for inp in match.split() if inp.strip()]
                # Remove type specifiers like 'wire' if present
                inputs = [inp for inp in inputs if not inp in ['input', 'wire']]
                # Remove inputs like ',' or ';'
                inputs = [inp for inp in inputs if not inp in [',', ';']]
                self.inputs.extend(inputs)

            # Process outputs
            output_matches = re.findall(r'output\s+([^;]+);', content)
            self.outputs = []
            for match in output_matches:
                # Split by spaces and filter out empty strings
                outputs = [out.strip() for out in match.split() if out.strip()]
                # Remove type specifiers like 'wire' if present
                outputs = [out for out in outputs if not out in ['output', 'wire']]
                # Remove outputs like ',' or ';'
                outputs = [out for out in outputs if not out in [',', ';']]
                self.outputs.extend(outputs)

            if self.debug:
                self.mensage.debug(f"Parsed module: {self.dut_name}")
                self.mensage.debug(f"Inputs: {self.inputs}")
                self.mensage.debug(f"Outputs: {self.outputs}")

        except FileNotFoundError:
            self.mensage.error(f"Module file not found: {self.module_path}")
            raise
        except Exception as e:
            self.mensage.error(f"Error parsing Verilog module: {str(e)}")
            raise

    def generate_basic_testbench(self) -> str:
        """
        Generate a basic Verilog testbench with exhaustive testing.

        Returns:
            str: Verilog testbench code
        """
        testbench_template = f"""/*
 * Testbench for {self.dut_name}
 * Generated automatically
 * Description: Exhaustive testing of all input combinations
 */

module {self.module_name}();
    // Testbench signals
{self._generate_testbench_signals()}

    // Error and test tracking
    integer errors = 0;
    integer tests_run = 0;

    // Device Under Test (DUT) instantiation
    {self.dut_name} dut(
{self._generate_dut_port_mapping()}
    );

    // Test stimulus and verification
    initial begin
        $display("Starting exhaustive test for {self.dut_name}");

        // Exhaustive testing of all input combinations
{self._generate_exhaustive_test_cases()}

        // Test summary
        $display("Test Summary:");
        $display("  Total Tests: %0d", tests_run);
        $display("  Errors Found: %0d", errors);

        if (errors == 0)
            $display("PASSED: All tests successful!");
        else
            $display("FAILED: %0d errors detected", errors);

        $finish;
    end

    // Optional: Waveform dumping
    initial begin
        $dumpfile("{self.module_name}.vcd");
        $dumpvars(0, {self.module_name});
    end

endmodule
"""
        return testbench_template

    def _generate_testbench_signals(self) -> str:
        """
        Generate signal declarations for testbench.

        Returns:
            str: Verilog signal declarations
        """
        signals = []
        for input_port in self.inputs:
            signals.append(f"    reg {input_port};")
        for output_port in self.outputs:
            signals.append(f"    wire {output_port};")
        return "\n".join(signals)

    def _generate_dut_port_mapping(self) -> str:
        """
        Generate port mapping for Device Under Test.

        Returns:
            str: Verilog port mapping
        """
        port_mappings = []
        for input_port in self.inputs:
            port_mappings.append(f"        .{input_port}({input_port})")
        for output_port in self.outputs:
            port_mappings.append(f"        .{output_port}({output_port})")
        return ",\n".join(port_mappings)

    def _generate_exhaustive_test_cases(self) -> str:
        """
        Generate exhaustive test cases for all possible input combinations.

        Returns:
            str: Verilog test case generation code
        """
        # Create test case generation using itertools
        test_cases = [
            "        // Exhaustive testing of all input combinations"
        ]

        # Generate all possible binary combinations
        def generate_combinations(num_inputs):
            self.mensage.debug(f"Generating all possible combinations for {num_inputs} inputs")
            return 2 ** num_inputs

        # Generate total input combinations
        input_combinations = generate_combinations(len(self.inputs))

        # Add test case generation code
        test_cases.append(f"        // Total test combinations: {input_combinations}")

        # Nested loops or direct generation of all combinations
        test_cases.append("        // Test every possible input combination")
        
        # Begin test loop
        test_cases.append("        begin")
        
        # Variables for iteration
        test_cases.append("            integer test_index;")
        test_cases.append(f"            for (test_index = 0; test_index < {input_combinations}; test_index = test_index + 1) begin")
        
        # Assign inputs based on combination
        for i, input_port in enumerate(self.inputs):
            test_cases.append(f"                {input_port} = {input_combinations}[test_index][{i}];")
        
        # Test tracking and delay
        test_cases.extend([
            "                tests_run = tests_run + 1;",
            "                #1;  // Allow signal propagation",
            "                // Add your specific verification logic here",
            "                // Example:",
            "                // if (output_doesnt_match_expected)",
            "                //     errors = errors + 1;"
        ])
        
        # Close loops
        test_cases.append("            end")
        test_cases.append("        end")

        return "\n".join(test_cases)

    def save_testbench(self, output_path: str = '.'):
        """
        Save generated testbench to a file.

        Args:
            output_path (str): Directory to save the testbench
        """
        try:
            # Generate testbench
            testbench_content = self.generate_basic_testbench()
            filename = f"testbench.v"

            # Save testbench file
            full_path = os.path.join(output_path, filename)
            with open(full_path, 'w') as f:
                f.write(testbench_content)

            if self.debug:
                self.mensage.debug(f"Saved testbench to {full_path}")
            self.mensage.success(f"Testbench saved: {filename}")

        except IOError as e:
            self.mensage.error(f"Failed to save testbench: {str(e)}")
            raise
        except Exception as e:
            self.mensage.error(f"Unexpected error: {str(e)}")
            raise

# Example usage
if __name__ == "__main__":
    generator = VerilogTestbenchGenerator("/app/results/somador_n2/verilog/somador.v", debug=True)
    generator.save_testbench()

[DEBUG] Parsed module: ripple_adder_2bit
[DEBUG] Inputs: ['a00', 'a01', 'b00', 'b01', 'cin']
[DEBUG] Outputs: ['sum00', 'sum01', 'cout']
[DEBUG] Initializing Testbench Generator for ripple_adder_2bit_tb
[DEBUG] Generating all possible combinations for 5 inputs
[DEBUG] Saved testbench to ./testbench.v
[SUCCESS] Testbench saved: testbench.v
